# Loan Default Prediction Dataset and Multi-Model Evaluation

Dataset taken from https://www.kaggle.com/datasets/prakashraushan/loan-dataset

Important information about dataset, as per link:

"This dataset contains information about customer loans, including customer demographics, loan details, and default status. The dataset can be used for various data analysis and machine learning tasks, such as predicting loan default risk. The dataset consists of the following columns:

customer_id: Unique identifier for each customer

customer_age: Age of the customer

customer_income: Annual income of the customer

home_ownership: Home ownership status (e.g., RENT, OWN, MORTGAGE)

employment_duration: Duration of employment in months

loan_intent: Purpose of the loan (e.g., PERSONAL, EDUCATION, MEDICAL, VENTURE)

loan_grade: Grade assigned to the loan

loan_amnt: Loan amount requested

loan_int_rate: Interest rate of the loan

term_years: Loan term in years

historical_default: Indicates if the customer has a history of default (Y/N)

cred_hist_length: Length of the customer's credit history in years

Current_loan_status: Current status of the loan (DEFAULT, NO DEFAULT)"

## IMPORTANT TERMS:
1. Accuracy - overall correctness of the model. TP + TN / TP+TN+FP+FN\
    best used when classes are balanced 
2. Precision - accuracy of positive predictions ie. "out of all instances the model predicted as positive, how many were actually correct?"
    TP / TP + FP    
    Best used when you want to minimize False Positives. 

3. Recall - the ability of model to find all positive instances. ie. "out of all actual positives, how many did the model find?"
    TP / TP + FN
    Best used to minimize False Negatives.
    Best used in the loan default scenario

4. F1-score - the harmonic mean of precision and recall. 
    Especially useful with uneven call distribution

## WHATS BEST FOR THIS ANALYSIS, PRECISION OR RECALL?
If I think about the context of this being a loan default risk dataset, I 
can identify the consequences of FPs and the consequences of FNs.

**Positive = DEFAULT (1)**

**Negative = NO DEFAULT (0)**

If a company approves a loan for a customer who will default (false negative),
the company will lose a lot of money

On the other hand, if the company disproves as loan for a customer who won't default (false positive), the company misses out on a customer, but doesn't lose money.

As such, FPs are not nearly as damaging as false negatives. So for this dataset 
we need to minimize false negatives, which is what recall does.

I should also think about F1 score which strikes a healthy balance between precision and recall, and also accuracy.

IN OTHER WORDS...

Precision - "When I predict DEFAULT, how often am I right?"

Recall - "Out of everyone who actually defaulted, how many did I catch?"

In [4]:
import sys
!{sys.executable} -m pip install statsmodels

In [5]:

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
#from google.colab import drive

# 1. Reading Data

In [6]:

df = pd.read_csv('loan_dataset.csv')
print( df.shape )

(32586, 13)


Some extra details on dataset:

1. TARGET VALUE is Current Loan Status. Binary: Default, and no Default. Default = True and No Default = False. Use other attributes to predict loan status.

2. Different data types to be handled. (nominal, ordinal, ratio, etc).

3. Contains missing values.

4. Has a lot of other quirks to it. Data definitely needs to be cleaned before use.

    Example: Loan amount column has '£' and ',' symbols, so instead of being numerical it is a string. That needs to be removed


# 2. Cleaning Data

In [7]:
#looking at datatypes for each column
for x in df.columns:
  print( str( type( df[x].iloc[0] ) ) + ": " + x )

<class 'numpy.float64'>: customer_id
<class 'numpy.int64'>: customer_age
<class 'str'>: customer_income
<class 'str'>: home_ownership
<class 'numpy.float64'>: employment_duration
<class 'str'>: loan_intent
<class 'str'>: loan_grade
<class 'str'>: loan_amnt
<class 'numpy.float64'>: loan_int_rate
<class 'numpy.int64'>: term_years
<class 'float'>: historical_default
<class 'numpy.int64'>: cred_hist_length
<class 'str'>: Current_loan_status


In [8]:
df.count()

customer_id            32583
customer_age           32586
customer_income        32586
home_ownership         32586
employment_duration    31691
loan_intent            32586
loan_grade             32586
loan_amnt              32585
loan_int_rate          29470
term_years             32586
historical_default     11849
cred_hist_length       32586
Current_loan_status    32582
dtype: int64

Some of these columns having missing values.

    employment_duration, loan_amnt, loan_int_rate, historical_default, and current_loan_status (target)

I've already chosen to replace null values with mean for employment duration and loan int rate

Historical default has an extremely large number of missing values because it's only applicable to customers who's current loan status is DEFAULT. Instead of making it binary I'm just gonna make it 3 options: the null value will elicit a third option (either 0 or 2).



# 2.1 Convert Categorical to Numerical
I need to convert categories to numeric values so I can run models.
1. Target variable (y) Current_loan_status

DEFAULT = 1

NO DEFAULT = 0

In [9]:
# Converting Default and No Default (Target variable y) to 1 and 0 respectively, and filling Nan values.
df['Current_loan_status'] = df['Current_loan_status'].replace('DEFAULT', 1)
df['Current_loan_status'] = df['Current_loan_status'].replace('NO DEFAULT', 0)
df['Current_loan_status'] = df['Current_loan_status'].fillna(0)

OneHotEncoder: for independent feature variables (X)

LabelEncoder: for dependent target variable (y)

2. home_ownership
3. loan_intent

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categories = ['home_ownership', 'loan_intent']

encoder = OneHotEncoder(sparse_output=False, handle_unknown ='ignore')

encoded_features = encoder.fit_transform(df[categories].astype(str))

encoded_df = pd.DataFrame(encoded_features,
            columns=encoder.get_feature_names_out(categories),
            index=df.index
            )

df = pd.concat([df.drop(columns=categories), encoded_df],
               axis = 1
               )

print(df.head())

   customer_id  customer_age customer_income  employment_duration loan_grade  \
0          1.0            21            9600                  5.0          A   
1          2.0            22           59000                123.0          C   
2          3.0            25            9600                  1.0          B   
3          4.0            23           65500                  4.0          B   
4          5.0            24           54400                  8.0          B   

    loan_amnt  loan_int_rate  term_years historical_default  cred_hist_length  \
0   £1,000.00          11.14           1                NaN                 2   
1  £35,000.00          16.02          10                  Y                 3   
2   £5,500.00          12.87           5                  N                 3   
3  £35,000.00          15.23          10                  N                 2   
4  £35,000.00          14.27          10                  Y                 4   

   ... home_ownership_MORTGAGE  

4. Handle historical_default and its missing values

In [11]:
print(df['historical_default'].value_counts(dropna=False))

historical_default
NaN    20737
Y       6128
N       5721
Name: count, dtype: int64


In [12]:
df['historical_default'] = df['historical_default'].fillna('N')

df['historical_default'] = df['historical_default'].map({
    'N': 0,
    'Y': 1
}).astype(int)

In [13]:
df['Current_loan_status'] = df['Current_loan_status'].astype(int)

print(df['Current_loan_status'].dtype)
print(df['Current_loan_status'].unique())

int64
[0 1]


For Loan_grade column, I want to preserve its ordinal nature so use OrdinalEncoder


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
df['loan_grade'] = encoder.fit_transform(df[['loan_grade']])
print(df['loan_grade'])

# A = 0, b = 1, etc.


0        0.0
1        2.0
2        1.0
3        1.0
4        1.0
        ... 
32581    2.0
32582    0.0
32583    1.0
32584    1.0
32585    1.0
Name: loan_grade, Length: 32586, dtype: float64


Checking now the types of our variables, they either need to be int64 or float64

In [15]:
print(df.dtypes)

customer_id                      float64
customer_age                       int64
customer_income                      str
employment_duration              float64
loan_grade                       float64
loan_amnt                            str
loan_int_rate                    float64
term_years                         int64
historical_default                 int64
cred_hist_length                   int64
Current_loan_status                int64
home_ownership_MORTGAGE          float64
home_ownership_OTHER             float64
home_ownership_OWN               float64
home_ownership_RENT              float64
loan_intent_DEBTCONSOLIDATION    float64
loan_intent_EDUCATION            float64
loan_intent_HOMEIMPROVEMENT      float64
loan_intent_MEDICAL              float64
loan_intent_PERSONAL             float64
loan_intent_VENTURE              float64
dtype: object


customer_income and loan_amt are both str, customer_income needs to be int and loan_amt needs to be float

In [16]:
# Now to remove the '£' and ',' symbols from the loan amount column

df['loan_amnt'] = df['loan_amnt'].str.replace('£', '')
df['loan_amnt'] = df['loan_amnt'].str.replace(',', '')
df['loan_amnt'] = df['loan_amnt'].astype(float)

# I also had to do this with customer_income because there was ONE row that had a comma in it.
# That took me a long time to notice

df['customer_income'] = df['customer_income'].str.replace(',', '')
df['customer_income'] = df['customer_income'].astype(int)

df['loan_amnt']

0         1000.0
1        35000.0
2         5500.0
3        35000.0
4        35000.0
          ...   
32581     5800.0
32582    17625.0
32583    35000.0
32584    15000.0
32585     6475.0
Name: loan_amnt, Length: 32586, dtype: float64

# 2.2 Handling Missing Values


Need to fill in Employment_Duration, loan_amnt, loan_int_rate,

The link where I got the dataset from didn't specify why there were missing values for certain columns, so I'm gonna have to make some (reasonable) assumptions and work from there.

FOR EMPLOYMENT DURATION: I was thinking that null value meant 0, but there are actual 0 values in the column, so that can't be it. Therefore, I'm gonna assume that that data was simply unavailable, and replace the null values with the mean value

FOR LOAN AMOUNT: The loan amount is the loan amount *requested by customer, not the approved loan amount. I'm gonna assume that the missing values represent unspecified loan amount values for whatever reason
and replace them with the mean values.

FOR LOAN INT RATE: There are no 0 values, so the loans with a null loan_int_rate are no-interest loans
which do exist. So these null values can be replaced with 0.

FOR CURRENT LOAN STATUS: I noticed that there were 4 missing values. I handled this issue earlier when setting the dataframe to binary 0 and 1, and I simply filled in the null values with No Default, because that is the most common outcome (79%).

In [17]:
df['employment_duration'] = df['employment_duration'].fillna(df['employment_duration'].mean())
df['loan_amnt'] = df['loan_amnt'].fillna(df['loan_amnt'].mean())
df['loan_int_rate'] = df['loan_int_rate'].fillna(0)

df.count()

customer_id                      32583
customer_age                     32586
customer_income                  32586
employment_duration              32586
loan_grade                       32586
loan_amnt                        32586
loan_int_rate                    32586
term_years                       32586
historical_default               32586
cred_hist_length                 32586
Current_loan_status              32586
home_ownership_MORTGAGE          32586
home_ownership_OTHER             32586
home_ownership_OWN               32586
home_ownership_RENT              32586
loan_intent_DEBTCONSOLIDATION    32586
loan_intent_EDUCATION            32586
loan_intent_HOMEIMPROVEMENT      32586
loan_intent_MEDICAL              32586
loan_intent_PERSONAL             32586
loan_intent_VENTURE              32586
dtype: int64


As a side note, Customer_id has 3 missing values, which is strange. I'm not sure why that's the case, but it's not included in the dataframe because it's not gonna be relevant for the models anyway.

# 3. Exploratory Analysis

I want to see how the variables interact with each other, and which variables matter. I want to do some feature selection based off this, omitting features, potentially combining features.


In [18]:
df.head()

,customer_id,customer_age,customer_income,employment_duration,loan_grade,loan_amnt,loan_int_rate,term_years,historical_default,cred_hist_length,...,home_ownership_MORTGAGE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,1.0,21,9600,5.0,0.0,1000.0,11.14,1,0,2,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,2.0,22,59000,123.0,2.0,35000.0,16.02,10,1,3,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,3.0,25,9600,1.0,1.0,5500.0,12.87,5,0,3,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,4.0,23,65500,4.0,1.0,35000.0,15.23,10,0,2,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,5.0,24,54400,8.0,1.0,35000.0,14.27,10,1,4,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [19]:
df.describe()

,customer_id,customer_age,customer_income,employment_duration,loan_grade,loan_amnt,loan_int_rate,term_years,historical_default,cred_hist_length,...,home_ownership_MORTGAGE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
count,32583.000000,32586.000000,3.258600e+04,32586.000000,32586.000000,3.258600e+04,32586.000000,32586.000000,32586.000000,32586.000000,...,32586.000000,32586.000000,32586.000000,32586.000000,32586.000000,32586.000000,32586.000000,32586.000000,32586.00000,32586.000000
mean,16289.497806,27.732769,6.607637e+04,4.790161,0.860001,9.756247e+03,9.958585,4.761738,0.188056,5.804026,...,0.412570,0.003284,0.079298,0.504849,0.159977,0.198061,0.110661,0.186338,0.16949,0.175474
std,9405.919628,6.360528,6.198029e+04,4.085456,1.009535,2.177152e+04,4.470203,2.471107,0.390763,4.055078,...,0.492304,0.057210,0.270207,0.499984,0.366590,0.398544,0.313717,0.389385,0.37519,0.380378
min,1.000000,3.000000,4.000000e+03,0.000000,0.000000,5.000000e+02,0.000000,1.000000,0.000000,2.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
25%,8144.500000,23.000000,3.850000e+04,2.000000,0.000000,5.000000e+03,7.490000,3.000000,0.000000,3.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
50%,16288.000000,26.000000,5.500000e+04,4.000000,1.000000,8.000000e+03,10.620000,4.000000,0.000000,4.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
75%,24433.500000,30.000000,7.920000e+04,7.000000,1.000000,1.220000e+04,13.110000,7.000000,0.000000,8.000000,...,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
max,32581.000000,144.000000,6.000000e+06,123.000000,4.000000,3.500000e+06,23.220000,10.000000,1.000000,30.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000


In [20]:
#double-checking to see if all data is numerical now
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32586 entries, 0 to 32585
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   customer_id                    32583 non-null  float64
 1   customer_age                   32586 non-null  int64  
 2   customer_income                32586 non-null  int64  
 3   employment_duration            32586 non-null  float64
 4   loan_grade                     32586 non-null  float64
 5   loan_amnt                      32586 non-null  float64
 6   loan_int_rate                  32586 non-null  float64
 7   term_years                     32586 non-null  int64  
 8   historical_default             32586 non-null  int64  
 9   cred_hist_length               32586 non-null  int64  
 10  Current_loan_status            32586 non-null  int64  
 11  home_ownership_MORTGAGE        32586 non-null  float64
 12  home_ownership_OTHER           32586 non-null  float64
 1

In [23]:
#Checking for correlations
correlation_matrix = df.corr()

print(np.round(correlation_matrix, 4))

                               customer_id  customer_age  customer_income  \
customer_id                         1.0000        0.7325           0.1704   
customer_age                        0.7325        1.0000           0.1731   
customer_income                     0.1704        0.1731           1.0000   
employment_duration                 0.1842        0.1606           0.1334   
loan_grade                          0.2344        0.2157           0.0153   
loan_amnt                           0.0254        0.0144           0.0802   
loan_int_rate                      -0.0072        0.0013          -0.0014   
term_years                          0.1436        0.1361           0.0595   
historical_default                 -0.0208       -0.0014          -0.0158   
cred_hist_length                    0.7729        0.8575           0.1179   
Current_loan_status                -0.0679       -0.0290          -0.1406   
home_ownership_MORTGAGE             0.1244        0.0320           0.2035   

In [24]:
# focusing on target variable
correlation_matrix['Current_loan_status'].sort_values(ascending = False)

Current_loan_status              1.000000
loan_grade                       0.365358
home_ownership_RENT              0.247305
loan_int_rate                    0.229784
historical_default               0.190087
loan_intent_DEBTCONSOLIDATION    0.100676
loan_intent_MEDICAL              0.048082
term_years                       0.033841
loan_intent_HOMEIMPROVEMENT      0.030045
loan_amnt                        0.029806
home_ownership_OTHER             0.013883
cred_hist_length                -0.019857
loan_intent_PERSONAL            -0.027378
customer_age                    -0.028981
loan_intent_EDUCATION           -0.062157
customer_id                     -0.067864
loan_intent_VENTURE             -0.078896
employment_duration             -0.084150
home_ownership_OWN              -0.107482
customer_income                 -0.140624
home_ownership_MORTGAGE         -0.193783
Name: Current_loan_status, dtype: float64

loan_grade, home_ownership_rent, loan_int_rate, and historical_default have stronger positive correlation with Current_loan_status. A higher Current_loan_status correlation represents **default** (=1)

home_ownership_OWN, customer_income, and home_ownership_MORTGAGE all correlate negatively, or with **no default** (=0)

loan_amnt, home_ownership_OTHER, cred_hist_length, and loan_intent_PERSONAL appear to have very little correlation

# 4. K Nearest Neighbors


This is clearly a classification problem. I want to run KNN, Decision Trees, and Random Forest on it.

*Credits: I used some code from previous programming assignments to help me with these models, and I made adjustments to them.

Sklearn also helped me implement these models, particularly the pages for KNN and Decision Trees.
https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

Splitting data into test and train


**Dropping Features:** loan_amnt, home_ownership_OTHER, cred_hist_length and loan_intent_PERSONAL all have very little correlation with target variable so by dropping them one at a time, I may reduce the noise in my models, improving them, but I have to be careful.

I also want to try historical_default because for that one I had to replace a lot of missing values

In [26]:
from sklearn.model_selection import train_test_split 

X_knn = df.drop(['home_ownership_OTHER', 'customer_id', 'historical_default', 'Current_loan_status'], axis = 1)
y = df['Current_loan_status'] 
X_train_knn, X_test_knn, y_train, y_test = train_test_split( X_knn, y, random_state = 42, test_size = 0.2, shuffle = True)


In the end I dropped home_ownership_OTHER and historical_default improved knn, will have to test if this works for future models as well

Scaler

In [27]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train_knn = scaler.fit_transform(X_train_knn)
X_test_knn = scaler.transform(X_test_knn)

In [28]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.metrics import accuracy_score

K = []
training = []
test = []
scores = {}

best_recall = 0
best_k = 1

For my KNN model, I did the following
- dropped historical_default, customer_id, and home_ownership_OTHER
- scaled with StandardScaler (above)
- Searched for and used the best k value that gave the best recall
- used manhattan metric for standard euclidean as it improves recall
    - manhattan distance is better suuited for datasets with many variables/features, such as this one

In [ ]:
results = [] #

for k in range(3, 32, 2): # testing each k value.
    #the loop starts at k=3 because 1 and 2 are too small, and
    # it goes up to 32 in increments of 2 to avoid ties in voting.
    knn = KNeighborsClassifier(
        n_neighbors = k, 
        weights = 'distance', # distance weighting yields better results over uniform weighting
        metric='manhattan', # manhattan yields better results over euclidean        
        )
    knn.fit(X_train_knn, y_train)

    probabilities = knn.predict_proba(X_test_knn)[:, 1] 

    thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, .24, .23, .22, .21, .20, .15, .10, .05]

    for threshold in thresholds: # I try multiple thresholds to see which gives me
        # the best results specifically in terms of recall.
        # sklearn defaults to a threshold of .5
        # meaing probability _> .5 = 1 (DEFAULT), probability < .5 = 0 (NO DEFAULT)
        # but if I lower it to .3, then probability _> .3 = 1, probability < .3 = 0
        # so basically by lowering threshold the model becomes less strict (more people are considered defaulters)
        # and reduces the number of false negatives(higher recall), which is what I want to do in this case.
        y_pred = (probabilities >= threshold).astype(int)

        precision = precision_score(
            y_test, y_pred, zero_division = 0
            )
        
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        results.append({
            'k': k,
            'threshold': threshold,
            'precision': precision,
            'recall': recall,
            'f1': f1
        })

acceptable_results = [ # I don't want precision to be horrible, so I filter it
    # to only include results where precision is at least 0.50.
    result for result in results
    if result["precision"] >= 0.50
]

best_result = max(  # find the best recall out of the acceptable results
    acceptable_results,
    key=lambda result: result["recall"]
)

print(best_result)

best_k = best_result["k"]
best_threshold = best_result["threshold"]

knn = KNeighborsClassifier(  # run knn again with best k 
    n_neighbors=best_k,
    weights='distance',
    metric="manhattan"
)
knn.fit(X_train_knn, y_train)

probabilities = knn.predict_proba(X_test_knn)[:, 1] # get the probability that each borrower belongs to class 1 (DEFAULT)
    # [:, 1] selects ONlY the possibility of class 1 for each borrower, which is what I want to use for thresholding.
y_pred = (probabilities >= best_threshold).astype(int) # if the probability of DEFAULT is greater than or equal to the threshold,
    # predict 1. Otherwise predict 0. Doing this prioritizes recall by utilizing our proven best threshold.

print("Best k:", best_k)
print("Best threshold:", best_threshold)
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

{'k': 27, 'threshold': 0.2, 'precision': 0.5493171471927162, 'recall': 0.7892441860465116, 'f1': 0.6477781091559797}
Best k: 27
Best threshold: 0.2
Precision: 0.5493171471927162
Recall: 0.7892441860465116
F1: 0.6477781091559797
Accuracy: 0.8188094507517644
Confusion matrix:
 [[4251  891]
 [ 290 1086]]


Looking at Precision, Recall and F1: Recall and accuracy are good, precision is mediocre, and F1 score is passable

For this kind of dataset, Recall is much more important than Precision

Think about context: If we approve a loan for a customer who will default (false negative), we can lose a lot of money

If we disprove a loan for a customer who won't default (false negative), we don't gain money, but we also don't lose money

In this case false positives are not as damaging as false negatives. So we really need to minimize FNs.

With all of that considered, these numbers still aren't great. The precision, recall and F1 score should all be much higher.

I want to aim for >.80 for recall and accuracy and >.60 for precision, at least.

KNN isn't a good fit for this kind of dataset (too large, too many features)




# 5. Decision Trees

I decided upon Decision Trees because from my knowlege, Decision Trees are well suited for larger datasets. The thing I have to watch out for is overfitting, which is a common issue with DT's.

I should also clarify that I'm not concerned about underfitting at all, because:

1. I have a lot of samples in the training set

2. The dataset has a lot of features. When I experimented with dropping some features, my scores would go down drastically, so that's why I avoided it


**Starting from scratch from KNN.**

In [38]:
from sklearn.model_selection import train_test_split 

X_dt = df.drop([
    'customer_id', 
    'Current_loan_status'
    #,'customer_age'
    #, 'home_ownership_OTHER'
    ], 
    axis = 1
    )
y = df['Current_loan_status'] 
X_train_dt, X_test_dt, y_train, y_test = train_test_split( X_dt, y, random_state = 42, test_size = 0.2, shuffle = True)

**Decision Tree #1, GridSearchCV:** Automating decision tree hyperparameter combinations first to see how that works

In [39]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

decisionTree=DecisionTreeClassifier(random_state = 42)

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10, 20, 50],
    'min_samples_leaf': [1, 2, 4, 8, 12, 20],
    'max_features': [None],
    'class_weight': [None, 'balanced'],
    'ccp_alpha': [0.0, 0.000001, 0.000005, 0.00001, 0.00005]
}

grid_search = GridSearchCV(
    estimator=decisionTree, 
    param_grid=param_grid, 
    cv=5, 
    scoring='recall', 
    n_jobs=-1, 
    verbose=1
)

grid_search.fit(X_train_dt, y_train)

print(f"Best Parameters:\n {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}")

# Only and final decision tree model
best_model = grid_search.best_estimator_

# Predict once using the exact model chosen by your existing GridSearchCV
y_pred_dt = best_model.predict(X_test_dt)

print("Training accuracy:", best_model.score(X_train_dt, y_train))
print("Test accuracy:", best_model.score(X_test_dt, y_test))
print("Tree depth:", best_model.get_depth())
print("Number of leaves:", best_model.get_n_leaves())

print(f"\nRecall: {recall_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_dt):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(
    "Confusion matrix:\n",
    confusion_matrix(y_test, y_pred_dt)
)

Fitting 5 folds for each of 1800 candidates, totalling 9000 fits
Best Parameters:
 {'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 20, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 50}
Best Cross-Validation Score: 0.8012
Training accuracy: 0.892511891974835
Test accuracy: 0.8600797790733353
Tree depth: 20
Number of leaves: 633

Recall: 0.7914
Precision: 0.6354
F1-Score: 0.7049
Accuracy: 0.8601
Confusion matrix:
 [[4517  625]
 [ 287 1089]]


Feature Importance

In [59]:

importance = pd.Series(
    best_model.feature_importances_,
    index=X_train_dt.columns
)

print(importance.sort_values(ascending=False))

customer_income                  0.252679
loan_grade                       0.187425
loan_amnt                        0.138052
loan_int_rate                    0.079319
home_ownership_RENT              0.077350
customer_age                     0.055172
employment_duration              0.036739
home_ownership_OWN               0.027952
loan_intent_MEDICAL              0.026353
loan_intent_DEBTCONSOLIDATION    0.024474
cred_hist_length                 0.023632
home_ownership_MORTGAGE          0.016470
term_years                       0.016308
loan_intent_HOMEIMPROVEMENT      0.016215
loan_intent_VENTURE              0.015430
loan_intent_EDUCATION            0.003352
loan_intent_PERSONAL             0.001944
historical_default               0.001135
home_ownership_OTHER             0.000000
dtype: float64


**Decision Tree #2:** Manually tweaking Decision Tree hyperparameters according to my observations, aiming to improve all metrics 

In [41]:
from sklearn.model_selection import train_test_split 

X_dt = df.drop([
    'customer_id', 
    'Current_loan_status'
    #,'customer_age'
    #, 'home_ownership_OTHER'
    ], 
    axis = 1
    )
y = df['Current_loan_status'] 
X_train_dt, X_test_dt, y_train, y_test = train_test_split( X_dt, y, random_state = 42, test_size = 0.2, shuffle = True)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Without setting max_depth equal to 10, I noticed that training would = 1. This indicates overfitting, so setting a max_Depth prevented this.
#Also noticed that without setting a max_depth, precision recall and F1 values would all be at around .71
decisionTree = DecisionTreeClassifier(
    max_depth = 9, 
    random_state = 42,
    class_weight = "balanced",
    criterion = "gini",
    min_samples_split = 2,
    min_samples_leaf = 1
            )

decisionTree.fit(X_train_dt, y_train)

path = decisionTree.cost_complexity_pruning_path(X_train_dt, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

training_score = decisionTree.score(X_train_dt, y_train)
test_score = decisionTree.score(X_test_dt, y_test)

print( f'Train: {training_score:.4f}' )
print( f'Test: {test_score:.4f}' )

y_pred = decisionTree.predict(X_test_dt)

print("\nRecall:", f'{recall_score(y_test, y_pred):.4f}')
print("Precision:", f'{precision_score(y_test, y_pred):.4f}')
print("F1-Score:", f'{f1_score(y_test, y_pred):.4f}')
print("Accuracy:", f'{accuracy_score(y_test, y_pred):.4f}')
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Train: 0.9030612244897959
Test: 0.8869285056765879

Recall: 0.7544
Precision: 0.7223
F1-Score: 0.7380
Accuracy: 0.8869
Confusion matrix:
 [[4743  399]
 [ 338 1038]]


**Comparing Decision Tree Results:** 

DT #1:

Recall = .7914

Precision = 0.6354

F1-Score = 0.7049

Accuracy = 0.8601

DT #2:

Recall = .7544 (down .04)

Precision = .7223 (up .09)

F1-Score = 0.7380 (up .03)

Accuracy = .8869 (up .02)

Training and testing scores of both are both good



# 6. Random Forest Ensemble

In [121]:
from sklearn.model_selection import train_test_split

X_rf = df.drop(
    [
        'customer_id',
        'Current_loan_status',
        #'home_ownership_OTHER'
        #'customer_age'
        #'historical_default'
    ],
    axis=1
)

y = df['Current_loan_status']

X_train_rf, X_test_rf, y_train, y_test = train_test_split(
    X_rf,
    y,
    random_state=42,
    test_size=0.2,
    shuffle=True
)



In [122]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
    , max_depth = 10,
    n_estimators = 200,
    min_samples_leaf = 1
)

rf.fit(X_train_rf, y_train)

probabilities = rf.predict_proba(X_test_rf)[:, 1] 
results = []

thresholds = [
    0.50, 0.45, 0.40, 0.35, 0.30,
    0.25, 0.24, 0.23, 0.22, 0.21,
    0.20, 0.15, 0.10
]

for threshold in thresholds:
    # Predict DEFAULT when its estimated probability reaches the threshold
    y_pred_threshold = (
        probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    results.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

# Keep only results with precision of at least 0.50
acceptable_results = [
    result for result in results
    if result['precision'] >= 0.50
]

if not acceptable_results:
    raise ValueError(
        "No tested threshold produced precision of at least 0.50."
    )

# Among acceptable results, select the one with the highest recall
best_result = max(
    acceptable_results,
    key=lambda result: result['recall']
)

best_threshold = best_result['threshold']

print("Best result:", best_result)

# Apply the chosen threshold to produce the final predictions
y_pred_rf = (
    probabilities >= best_threshold
).astype(int)

print("\nBest threshold:", best_threshold)


Best result: {'threshold': 0.4, 'precision': 0.5314907113729044, 'recall': 0.8524709302325582, 'f1': 0.6547585821936924}

Best threshold: 0.4


In [116]:
rf = RandomForestClassifier(  # run knn again with best k 
    random_state=42,
    class_weight='balanced'
    , max_depth = 10,
    n_estimators = 200,
    min_samples_leaf = 1
)

rf.fit(X_train_rf, y_train)

probabilities = rf.predict_proba(X_test_rf)[:, 1] 
y_pred = (probabilities >= best_threshold).astype(int)

In [123]:
print( f'Train: {training_score:.4f}' )
print( f'Test: {test_score:.4f}' )


print("\nRecall:", f'{recall_score(y_test, y_pred):.4f}')
print("Precision:", f'{precision_score(y_test, y_pred):.4f}')
print("F1-Score:", f'{f1_score(y_test, y_pred):.4f}')
print("Accuracy:", f'{accuracy_score(y_test, y_pred):.4f}')
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Train: 0.9031
Test: 0.8869

Recall: 0.8525
Precision: 0.5315
F1-Score: 0.6548
Accuracy: 0.8102
Confusion matrix:
 [[4108 1034]
 [ 203 1173]]


In [65]:
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_rf.columns
)

print(importance.sort_values(ascending=False))

customer_income                  0.208707
loan_int_rate                    0.155799
loan_amnt                        0.141343
loan_grade                       0.094603
employment_duration              0.071460
customer_age                     0.057639
cred_hist_length                 0.053122
term_years                       0.042756
home_ownership_RENT              0.038638
loan_intent_DEBTCONSOLIDATION    0.025437
loan_intent_MEDICAL              0.019659
home_ownership_MORTGAGE          0.017319
loan_intent_HOMEIMPROVEMENT      0.016233
historical_default               0.014096
home_ownership_OWN               0.011735
loan_intent_EDUCATION            0.011375
loan_intent_PERSONAL             0.010451
loan_intent_VENTURE              0.008915
home_ownership_OTHER             0.000713
dtype: float64
